In [6]:
# ==========================================
# UJI MANN-WHITNEY U
# DATA SEIMBANG 335 SAMPLE PER MODEL
# HASIL LEBIH STABIL UNTUK VISUALISASI
# ==========================================

# ==========================================
# 1. IMPORT LIBRARY
# ==========================================
from pathlib import Path
import pandas as pd
from scipy.stats import mannwhitneyu

# ==========================================
# 2. LOAD DATASET
# ==========================================
base = Path(".")

codellama = pd.read_excel(
    base / "Hasil_CodeLlama_GitHub_fix.xlsx"
)

deepseek = pd.read_excel(
    base / "Hasil_Deepseek_GitHub_fix.xlsx"
)

qwen = pd.read_excel(
    base / "Hasil_Qwen_GitHub_fix.xlsx"
)

# ==========================================
# 3. AMBIL SAMPLE SEIMBANG
# ==========================================
codellama = codellama.sample(
    n=2357,
    random_state=335
)

deepseek = deepseek.sample(
    n=2357,
    random_state=335
)

qwen = qwen.sample(
    n=2357,
    random_state=335
)

# ==========================================
# 4. TAMBAHKAN LABEL MODEL
# ==========================================
codellama["Model"] = "CodeLlama"
deepseek["Model"] = "DeepSeek"
qwen["Model"] = "Qwen"

# ==========================================
# 5. GABUNGKAN DATA
# ==========================================
df = pd.concat(
    [codellama, deepseek, qwen],
    ignore_index=True
)
# ==========================================
# KONVERSI SELURUH KOLOM MENJADI NUMERIK
# ==========================================

numeric_columns = [

    "BLEU Zero",
    "ROUGE Zero",
    "METEOR Zero",

    "BLEU Few",
    "ROUGE Few",
    "METEOR Few",

    "BLEU Adv",
    "ROUGE Adv",
    "METEOR Adv"
]

for col in numeric_columns:

    # ubah koma menjadi titik
    df[col] = df[col].astype(str).str.replace(",", ".")

    # konversi ke numerik
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )
# ==========================================
# 6. STRATEGI PROMPTING
# ==========================================
strategies = {

    "Zero-shot": [
        "BLEU Zero",
        "ROUGE Zero",
        "METEOR Zero"
    ],

    "Few-shot": [
        "BLEU Few",
        "ROUGE Few",
        "METEOR Few"
    ],

    "Chain of Thought": [
        "BLEU Adv",
        "ROUGE Adv",
        "METEOR Adv"
    ]
}

# ==========================================
# 7. PASANGAN PERBANDINGAN
# ==========================================
comparisons = [

    ("Zero-shot", "Few-shot"),

    ("Zero-shot", "Chain of Thought"),

    ("Few-shot", "Chain of Thought")
]

# ==========================================
# 8. LIST HASIL
# ==========================================
results = []

# ==========================================
# 9. LOOPING PENGUJIAN
# ==========================================
for model in [

    "CodeLlama",
    "DeepSeek",
    "Qwen"
]:

    # filter model
    model_df = df[
        df["Model"] == model
    ]

    # looping metrik
    for metric_name, idx in zip(

        ["BLEU", "ROUGE", "METEOR"],

        [0, 1, 2]
    ):

        # looping perbandingan
        for s1, s2 in comparisons:

            # ==================================
            # AMBIL KOLOM
            # ==================================
            col1 = strategies[s1][idx]
            col2 = strategies[s2][idx]

            # ==================================
            # AMBIL DATA
            # ==================================
            data1 = model_df[col1].dropna()
            data2 = model_df[col2].dropna()

            # ==================================
            # UJI MANN-WHITNEY U
            # ==================================
            stat, p = mannwhitneyu(
                data1,
                data2,
                alternative='two-sided'
            )

            # ==================================
            # HITUNG RATA-RATA
            # ==================================
            mean1 = data1.mean()
            mean2 = data2.mean()

            # ==================================
            # NORMALISASI P-VALUE
            # AGAR LEBIH MUDAH DIBACA
            # ==================================
            normalized_p = max(
                p,
                0.001
            )

            # ==================================
            # STRATEGI DENGAN NILAI TERTINGGI
            # ==================================
            better = s1 if mean1 > mean2 else s2

            # ==================================
            # HASIL SIGNIFIKANSI
            # ==================================
            significance = (
                "Ya"
                if normalized_p < 0.05
                else "Tidak"
            )

            # ==================================
            # SIMPAN HASIL
            # ==================================
            results.append({

                "Model":
                    model,

                "Metrik":
                    metric_name,

                "Perbandingan":
                    f"{s1} vs {s2}",

                "Mean 1":
                    round(mean1, 4),

                "Mean 2":
                    round(mean2, 4),

                "U-Statistic":
                    round(stat, 4),

                "P-Value":
                    round(normalized_p, 4),

                "Signifikan":
                    significance,

                "Hasil Lebih Tinggi":
                    better
            })

# ==========================================
# 10. UBAH KE DATAFRAME
# ==========================================
results_df = pd.DataFrame(results)

# ==========================================
# 11. TAMPILKAN HASIL
# ==========================================
print(results_df)

# ==========================================
# 12. SIMPAN KE EXCEL
# ==========================================
results_df.to_excel(
    "Hasil_Uji_Mann_Whitney_335.xlsx",
    index=False
)

print(
    "\nHasil pengujian berhasil disimpan."
)

        Model  Metrik                   Perbandingan  Mean 1  Mean 2  \
0   CodeLlama    BLEU          Zero-shot vs Few-shot  0.0040  0.0040   
1   CodeLlama    BLEU  Zero-shot vs Chain of Thought  0.0040  0.0075   
2   CodeLlama    BLEU   Few-shot vs Chain of Thought  0.0040  0.0075   
3   CodeLlama   ROUGE          Zero-shot vs Few-shot  0.0643  0.0778   
4   CodeLlama   ROUGE  Zero-shot vs Chain of Thought  0.0643  0.0580   
5   CodeLlama   ROUGE   Few-shot vs Chain of Thought  0.0778  0.0580   
6   CodeLlama  METEOR          Zero-shot vs Few-shot  0.1068  0.1263   
7   CodeLlama  METEOR  Zero-shot vs Chain of Thought  0.1068  0.0968   
8   CodeLlama  METEOR   Few-shot vs Chain of Thought  0.1263  0.0968   
9    DeepSeek    BLEU          Zero-shot vs Few-shot  0.0036  0.0035   
10   DeepSeek    BLEU  Zero-shot vs Chain of Thought  0.0036  0.0036   
11   DeepSeek    BLEU   Few-shot vs Chain of Thought  0.0035  0.0036   
12   DeepSeek   ROUGE          Zero-shot vs Few-shot  0.0583  0.

In [4]:
print(df[numeric_columns].dtypes)

BLEU Zero      float64
ROUGE Zero     float64
METEOR Zero    float64
BLEU Few       float64
ROUGE Few      float64
METEOR Few     float64
BLEU Adv       float64
ROUGE Adv      float64
METEOR Adv     float64
dtype: object


In [7]:
# ==========================================
# VISUALISASI HASIL MANN-WHITNEY U
# MENGGUNAKAN NILAI RATA-RATA
# AGAR TIDAK ADA 0.0000
# ==========================================

# ==========================================
# 1. IMPORT LIBRARY
# ==========================================
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 2. LOAD DATASET
# ==========================================
base = Path(".")

codellama = pd.read_excel(
    base / "Hasil_CodeLlama_GitHub_fix.xlsx"
)

deepseek = pd.read_excel(
    base / "Hasil_Deepseek_GitHub_fix.xlsx"
)

qwen = pd.read_excel(
    base / "Hasil_Qwen_GitHub_fix.xlsx"
)

# ==========================================
# 3. AMBIL SAMPLE SEIMBANG
# ==========================================
codellama = codellama.sample(
    n=2578,
    random_state=335
)

deepseek = deepseek.sample(
    n=2578,
    random_state=335
)

qwen = qwen.sample(
    n=2578,
    random_state=335
)

# ==========================================
# 4. TAMBAHKAN LABEL MODEL
# ==========================================
codellama["Model"] = "CodeLlama"
deepseek["Model"] = "DeepSeek"
qwen["Model"] = "Qwen"

# ==========================================
# 5. GABUNGKAN DATA
# ==========================================
df = pd.concat(
    [codellama, deepseek, qwen],
    ignore_index=True
)
# ==========================================
# KONVERSI SELURUH KOLOM MENJADI NUMERIK
# ==========================================

numeric_columns = [

    "BLEU Zero",
    "ROUGE Zero",
    "METEOR Zero",

    "BLEU Few",
    "ROUGE Few",
    "METEOR Few",

    "BLEU Adv",
    "ROUGE Adv",
    "METEOR Adv"
]

for col in numeric_columns:

    # ubah koma menjadi titik
    df[col] = df[col].astype(str).str.replace(",", ".")

    # konversi ke numerik
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# ==========================================
# 6. STRATEGI PROMPTING
# ==========================================
strategies = {

    "Zero-shot": [
        "BLEU Zero",
        "ROUGE Zero",
        "METEOR Zero"
    ],

    "Few-shot": [
        "BLEU Few",
        "ROUGE Few",
        "METEOR Few"
    ],

    "Chain of Thought": [
        "BLEU Adv",
        "ROUGE Adv",
        "METEOR Adv"
    ]
}

# ==========================================
# 7. LOOPING VISUALISASI
# ==========================================
for model in [
    "CodeLlama",
    "DeepSeek",
    "Qwen"
]:

    model_df = df[
        df["Model"] == model
    ]

    matrix = []

    metrics = [
        "BLEU",
        "ROUGE",
        "METEOR"
    ]

    for idx in [0, 1, 2]:

        row = []

        for strategy in strategies:

            col = strategies[strategy][idx]

            mean_value = model_df[col].mean()

            row.append(mean_value)

        matrix.append(row)

    matrix = np.array(matrix)

    # ======================================
    # FIGURE HD
    # ======================================
    fig, ax = plt.subplots(
        figsize=(10, 7),
        dpi=300
    )

    # ======================================
    # HEATMAP
    # ======================================
    heatmap = ax.imshow(
        matrix,
        cmap="Pastel2"
    )

    # ======================================
    # LABEL
    # ======================================
    ax.set_xticks(np.arange(3))
    ax.set_yticks(np.arange(3))

    ax.set_xticklabels(
        [
            "Zero-shot",
            "Few-shot",
            "Chain of Thought"
        ],
        fontsize=12,
        fontweight="bold"
    )

    ax.set_yticklabels(
        metrics,
        fontsize=12,
        fontweight="bold"
    )

    # ======================================
    # ANGKA DI TENGAH
    # ======================================
    for i in range(3):
        for j in range(3):

            value = matrix[i, j]

            ax.text(
                j,
                i,
                f"{value:.4f}",
                ha="center",
                va="center",
                fontsize=12,
                fontweight="bold",
                color="black"
            )

    # ======================================
    # TITLE
    # ======================================
    ax.set_title(
        f"Perbandingan Nilai Evaluasi\n{model}",
        fontsize=18,
        fontweight="bold",
        pad=20
    )

    # ======================================
    # GRID
    # ======================================
    ax.set_xticks(
        np.arange(-.5, 3, 1),
        minor=True
    )

    ax.set_yticks(
        np.arange(-.5, 3, 1),
        minor=True
    )

    ax.grid(
        which="minor",
        color="white",
        linestyle="-",
        linewidth=3
    )

    # ======================================
    # HILANGKAN BORDER
    # ======================================
    for edge in ax.spines.values():
        edge.set_visible(False)

    # ======================================
    # COLORBAR
    # ======================================
    cbar = plt.colorbar(heatmap)

    cbar.ax.tick_params(
        labelsize=10
    )

    # ======================================
    # RAPIIKAN
    # ======================================
    plt.tight_layout()

    # ======================================
    # SIMPAN HD
    # ======================================
    plt.savefig(
        f"Heatmap_{model}.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close()

print(
    "\nVisualisasi berhasil dibuat."
)


Visualisasi berhasil dibuat.


In [9]:
# ==========================================
# VISUALISASI PERBANDINGAN
# ANTAR MODEL LLM
# ==========================================

# ==========================================
# 1. IMPORT LIBRARY
# ==========================================
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 2. LOAD DATASET
# ==========================================
base = Path(".")

codellama = pd.read_excel(
    base / "Hasil_CodeLlama_GitHub_fix.xlsx"
)

deepseek = pd.read_excel(
    base / "Hasil_Deepseek_GitHub_fix.xlsx"
)

qwen = pd.read_excel(
    base / "Hasil_Qwen_GitHub_fix.xlsx"
)

# ==========================================
# 3. AMBIL SAMPLE SEIMBANG
# ==========================================
codellama = codellama.sample(
    n=335,
    random_state=42
)

deepseek = deepseek.sample(
    n=335,
    random_state=42
)

qwen = qwen.sample(
    n=335,
    random_state=42
)

# ==========================================
# 4. TAMBAHKAN LABEL MODEL
# ==========================================
codellama["Model"] = "CodeLlama"
deepseek["Model"] = "DeepSeek"
qwen["Model"] = "Qwen"

# ==========================================
# 5. GABUNGKAN DATA
# ==========================================
df = pd.concat(
    [codellama, deepseek, qwen],
    ignore_index=True
)

# ==========================================
# 6. HITUNG RATA-RATA
# ==========================================
results = []

models = [
    "CodeLlama",
    "DeepSeek",
    "Qwen"
]

strategies = {

    "Zero-shot": [
        "BLEU Zero",
        "ROUGE Zero",
        "METEOR Zero"
    ],

    "Few-shot": [
        "BLEU Few",
        "ROUGE Few",
        "METEOR Few"
    ],

    "Chain of Thought": [
        "BLEU Adv",
        "ROUGE Adv",
        "METEOR Adv"
    ]
}

# ==========================================
# 7. LOOPING RATA-RATA
# ==========================================
for model in models:

    model_df = df[
        df["Model"] == model
    ]

    for strategy, cols in strategies.items():

        bleu = model_df[
            cols[0]
        ].mean()

        rouge = model_df[
            cols[1]
        ].mean()

        meteor = model_df[
            cols[2]
        ].mean()

        overall = (
            bleu +
            rouge +
            meteor
        ) / 3

        results.append({

            "Model":
                model,

            "Strategy":
                strategy,

            "BLEU":
                round(bleu, 4),

            "ROUGE":
                round(rouge, 4),

            "METEOR":
                round(meteor, 4),

            "Overall":
                round(overall, 4)
        })

# ==========================================
# 8. DATAFRAME HASIL
# ==========================================
results_df = pd.DataFrame(results)

# ==========================================
# 9. VISUALISASI BAR CHART
# ==========================================
fig, ax = plt.subplots(
    figsize=(14, 7),
    dpi=300
)

labels = []
values = []

for _, row in results_df.iterrows():

    labels.append(
        f"{row['Model']}\n{row['Strategy']}"
    )

    values.append(
        row["Overall"]
    )

# ==========================================
# WARNA TERANG
# ==========================================
colors = [

    "#A8DADC",
    "#BDE0FE",
    "#CDB4DB",

    "#FFD6A5",
    "#FFCAD4",
    "#D8E2DC",

    "#E9C46A",
    "#F4A261",
    "#B8F2E6"
]

# ==========================================
# BAR CHART
# ==========================================
bars = ax.bar(
    labels,
    values,
    color=colors
)

# ==========================================
# TAMBAHKAN ANGKA
# ==========================================
for bar, value in zip(
    bars,
    values
):

    ax.text(
        bar.get_x() + bar.get_width()/2,
        value,
        f"{value:.4f}",
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold',
        color='black'
    )

# ==========================================
# TITLE
# ==========================================
ax.set_title(
    "Perbandingan Kualitas Ringkasan Kode Antar Model",
    fontsize=18,
    fontweight="bold",
    pad=20
)

# ==========================================
# LABEL Y
# ==========================================
ax.set_ylabel(
    "Rata-rata Nilai Evaluasi",
    fontsize=12,
    fontweight="bold"
)

# ==========================================
# GRID
# ==========================================
ax.grid(
    axis='y',
    linestyle='--',
    alpha=0.3
)

# ==========================================
# ROTASI LABEL
# ==========================================
plt.xticks(
    rotation=10,
    fontsize=10,
    fontweight="bold"
)

plt.yticks(
    fontsize=10
)

# ==========================================
# RAPIIKAN
# ==========================================
plt.tight_layout()

# ==========================================
# SIMPAN HD
# ==========================================
plt.savefig(
    "Perbandingan_Antar_Model_HD.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.close()

# ==========================================
# 10. HEATMAP PERBANDINGAN
# ==========================================
heatmap_data = []

for model in models:

    row = []

    for strategy in strategies:

        value = results_df[
            (results_df["Model"] == model)
            &
            (results_df["Strategy"] == strategy)
        ]["Overall"].values[0]

        row.append(value)

    heatmap_data.append(row)

heatmap_data = np.array(
    heatmap_data
)

# ==========================================
# FIGURE HEATMAP
# ==========================================
fig, ax = plt.subplots(
    figsize=(10, 7),
    dpi=300
)

# ==========================================
# HEATMAP
# ==========================================
heatmap = ax.imshow(
    heatmap_data,
    cmap="Pastel2"
)

# ==========================================
# LABEL
# ==========================================
ax.set_xticks(np.arange(3))
ax.set_yticks(np.arange(3))

ax.set_xticklabels(
    [
        "Zero-shot",
        "Few-shot",
        "Chain of Thought"
    ],
    fontsize=12,
    fontweight="bold"
)

ax.set_yticklabels(
    models,
    fontsize=12,
    fontweight="bold"
)

# ==========================================
# ANGKA
# ==========================================
for i in range(3):
    for j in range(3):

        value = heatmap_data[i, j]

        ax.text(
            j,
            i,
            f"{value:.4f}",
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold",
            color="black"
        )

# ==========================================
# TITLE
# ==========================================
ax.set_title(
    "Heatmap Perbandingan Antar Model",
    fontsize=18,
    fontweight="bold",
    pad=20
)

# ==========================================
# GRID
# ==========================================
ax.set_xticks(
    np.arange(-.5, 3, 1),
    minor=True
)

ax.set_yticks(
    np.arange(-.5, 3, 1),
    minor=True
)

ax.grid(
    which="minor",
    color="white",
    linestyle="-",
    linewidth=3
)

# ==========================================
# HILANGKAN BORDER
# ==========================================
for edge in ax.spines.values():
    edge.set_visible(False)

# ==========================================
# COLORBAR
# ==========================================
cbar = plt.colorbar(
    heatmap
)

cbar.ax.tick_params(
    labelsize=10
)

# ==========================================
# RAPIIKAN
# ==========================================
plt.tight_layout()

# ==========================================
# SIMPAN HD
# ==========================================
plt.savefig(
    "Heatmap_Perbandingan_Model_HD.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.close()

print(
    "\nVisualisasi berhasil dibuat."
)

print(
    "\nFile yang dihasilkan:"
)

print("- Perbandingan_Antar_Model_HD.png")
print("- Heatmap_Perbandingan_Model_HD.png")


Visualisasi berhasil dibuat.

File yang dihasilkan:
- Perbandingan_Antar_Model_HD.png
- Heatmap_Perbandingan_Model_HD.png
